In [ ]:
#EXTRACT TEXT FROM PDF

import os
import pymupdf
import re

os.makedirs("./textDataFil_pymupdf", exist_ok=True)
# print(os.listdir("./booksPDF"))

pdfContent = {}

for i in os.listdir("./booksPDF"):
    pdfText = ""
    filePath = f"./booksPDF/{i}"
    
    doc = pymupdf.open(filePath)

    for pageNum in range(1, doc.page_count):  
        currentPage = doc[pageNum]
        currentText = currentPage.get_text()

        # Skip unwanted sections
        if any(skip_phrase in currentText for skip_phrase in [
            "Brought to you by", "International License", 
            "Table of Contents", "This book was developed as part of the ABC",
            "Advancing Basic Education in the Philippines",
            "licensed under the Creative Commons",
            "Attribution-NonCommercial",
            "If you create an adaptation of this work",
            "please use the following label on your work",
            "developed under the USAID ABC+: Advancing Basic Education in the Philippines project"
        ]):
            continue  

        # Normalize curly quotes to straight quotes
        currentText = currentText.replace("“", '"').replace("”", '"') \
                             .replace("‘", "'").replace("’", "'")

        # Remove standalone numbers
        currentText = re.sub(r'\b\d+\b', '', currentText)

        pdfText += currentText.strip()
    pdfContent[i] = pdfText

In [36]:
#STORE EXTRACTED TEXT INTO TXT FILES

for key in pdfContent.keys():
    key.split(".")
    processedKey = re.sub(r'[\.]','',key)
    processedKey = re.sub(r' ','_',processedKey)
    processedKey = re.sub(r'\?','',processedKey)
    processedKey = re.sub(r'\n','_',processedKey)
    # print(pdfContent[key])

    f = open(f"./textDataFil_pymupdf/{processedKey}.txt","w",encoding='utf-8')
    f.writelines(pdfContent[key])
    f.close()

In [37]:
#CONCATENATING ALL TXT FILE TOGETHER

sourceFolder = "./textDataFil_pymupdf"

totalText = ""

for i in os.listdir(sourceFolder):
    filePathForTokenizingText = f"{sourceFolder}/{i}"
    f = open(filePathForTokenizingText,'r', encoding='utf-8')
    insideText = f.read()

    insideText = re.sub(r'[.]','. ',insideText)
    insideText = re.sub('\n',' ',insideText)
    insideText = re.sub('- ','-',insideText)
    insideText = re.sub('  ',' ',insideText)
    insideText = re.sub('&nbsp;','',insideText)

    totalText += insideText
    totalText += "\n\n"

f2 = open("./textData_forTokenization/textAllFil.txt", "w", encoding='utf-8')
f2.writelines(totalText)
f2.close()